In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import QuantileTransformer
from sklearn.linear_model import LogisticRegression

# --- Veri ---
df = pd.read_csv('spam.csv', encoding='latin-1')
df.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)
df.columns = ['type', 'sms']
df['type'] = df['type'].map({'ham': 0, 'spam': 1})

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' url ', text)
    text = re.sub(r'\d+', ' num ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['sms_clean'] = df['sms'].apply(preprocess_text)
df['sms_original'] = df['sms']  # orijinali sakla
df = df[df['sms_clean'] != ''].reset_index(drop=True)

X = df['sms_clean']
y = df['type']
X_orig = df['sms_original']

X_train, X_test, y_train, y_test, Xo_train, Xo_test = train_test_split(
    X, y, X_orig, test_size=0.3, random_state=111, stratify=y
)

# --- Model ---
vec = CountVectorizer()
X_train_vec = vec.fit_transform(X_train).toarray()
X_test_vec  = vec.transform(X_test).toarray()

scaler = QuantileTransformer(output_distribution='normal', random_state=42)
X_train_s = scaler.fit_transform(X_train_vec)
X_test_s  = scaler.transform(X_test_vec)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

y_pred  = model.predict(X_test_s)
y_prob  = model.predict_proba(X_test_s)[:, 1]

# --- Hatalı sınıflandırmalar ---
results = pd.DataFrame({
    'original_sms': Xo_test.values,
    'clean_sms':    X_test.values,
    'true_label':   y_test.values,
    'pred_label':   y_pred,
    'spam_prob':    y_prob.round(4),
})

# False Negatives: gerçek spam ama ham tahmin edilmiş
FN = results[(results['true_label']==1) & (results['pred_label']==0)].copy()
FN = FN.sort_values('spam_prob')  # en düşük spam olasılıklıdan başla

# False Positives: gerçek ham ama spam tahmin edilmiş
FP = results[(results['true_label']==0) & (results['pred_label']==1)].copy()
FP = FP.sort_values('spam_prob', ascending=False)

print(f"Toplam test: {len(results)}")
print(f"False Negatives (kaçırılan spam): {len(FN)}")
print(f"False Positives (yanlış alarm):   {len(FP)}")
print(f"\n--- FALSE NEGATIVES (spam_prob düşükten yükseğe) ---")
pd.set_option('display.max_colwidth', 120)
print(FN[['original_sms', 'spam_prob']].to_string())
print(f"\n--- FALSE POSITIVES ---")
print(FP[['original_sms', 'spam_prob']].to_string())

Toplam test: 1671
False Negatives (kaçırılan spam): 18
False Positives (yanlış alarm):   1

--- FALSE NEGATIVES (spam_prob düşükten yükseğe) ---
                                                                                                                                                          original_sms  spam_prob
808                                                                               Would you like to see my XXX pics they are so hot they were nearly banned in the uk!     0.0000
689                                 Hello darling how are you today? I would love to have a chat, why dont you tell me what you look like and what you are in to sexy?     0.0000
1524                                                                                            Latest News! Police station toilet stolen, cops have nothing to go on!     0.0001
700                                                                                   Did you hear about the new \Divorce Barbie\"? It comes wi

In [2]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam'], digits=4))

[[1446    1]
 [  18  206]]

              precision    recall  f1-score   support

         Ham     0.9877    0.9993    0.9935      1447
        Spam     0.9952    0.9196    0.9559       224

    accuracy                         0.9886      1671
   macro avg     0.9914    0.9595    0.9747      1671
weighted avg     0.9887    0.9886    0.9884      1671



In [3]:
import numpy as np
from scipy.stats import wilcoxon
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import QuantileTransformer, Binarizer
from sklearn.feature_extraction.text import CountVectorizer

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Vektörleştir
vec = CountVectorizer()
X_bow = vec.fit_transform(X_train).toarray()

# Model 1: BoW + QuantileTransformer + LR
qt = QuantileTransformer(output_distribution='normal', random_state=42)
X_qt = qt.fit_transform(X_bow)
scores_1 = cross_val_score(LogisticRegression(max_iter=1000), X_qt, y_train, cv=cv, scoring='accuracy')
print(f"BoW+QT+LR:  {scores_1} → mean={scores_1.mean():.4f}")

# Model 2: BoW + Binarizer + LR
X_bin = Binarizer().fit_transform(X_bow)
scores_2 = cross_val_score(LogisticRegression(max_iter=1000), X_bin, y_train, cv=cv, scoring='accuracy')
print(f"BoW+Bin+LR: {scores_2} → mean={scores_2.mean():.4f}")

# Model 3: BoW + Binarizer + SVM
scores_3 = cross_val_score(SVC(kernel='linear', probability=True), X_bin, y_train, cv=cv, scoring='accuracy')
print(f"BoW+Bin+SVM:{scores_3} → mean={scores_3.mean():.4f}")

# Wilcoxon testleri
stat, p = wilcoxon(scores_1, scores_2)
print(f"\nBoW+QT+LR vs BoW+Bin+LR:  stat={stat:.4f}, p={p:.4f}")
stat, p = wilcoxon(scores_1, scores_3)
print(f"BoW+QT+LR vs BoW+Bin+SVM: stat={stat:.4f}, p={p:.4f}")
stat, p = wilcoxon(scores_2, scores_3)
print(f"BoW+Bin+LR vs BoW+Bin+SVM: stat={stat:.4f}, p={p:.4f}")

BoW+QT+LR:  [0.98589744 0.98974359 0.98589744 0.98846154 0.98459564] → mean=0.9869
BoW+Bin+LR: [0.98846154 0.98974359 0.98461538 0.98589744 0.98202824] → mean=0.9861
BoW+Bin+SVM:[0.98461538 0.98717949 0.98589744 0.98333333 0.98074454] → mean=0.9844

BoW+QT+LR vs BoW+Bin+LR:  stat=2.5000, p=0.5000
BoW+QT+LR vs BoW+Bin+SVM: stat=0.0000, p=0.1250
BoW+Bin+LR vs BoW+Bin+SVM: stat=1.0000, p=0.1250
